# 🐱 🐶 A mini CNN project about cats and dogs...

## 📚 Libraries

In [ ]:
# DATA MANIPULATION
import numpy as np
# DATA VIZ
import matplotlib.pyplot as plt

# IMAGE PROCESSING AND LOADER
from keras.preprocessing.image import load_img, img_to_array
from keras.preprocessing import image_dataset_from_directory

# NEURAL NETWORKS
from keras import models,layers, optimizers, callbacks

# PRE-TRAINED MODEL
from keras.applications.vgg16 import VGG16

## 🛠️ Utils

In [ ]:
def plot_history(history):
    fig, ax = plt.subplots(1, 2, figsize=(15,5))
    ax[0].set_title('loss')
    ax[0].plot(history.epoch, history.history["loss"], label="Train loss")
    ax[0].plot(history.epoch, history.history["val_loss"], label="Validation loss")
    ax[1].set_title('accuracy')
    ax[1].plot(history.epoch, history.history["accuracy"], label="Train acc")
    ax[1].plot(history.epoch, history.history["val_accuracy"], label="Validation acc")
    ax[0].legend()
    ax[1].legend()

In [ ]:
def plot_compare_history(history, name_history, history_1, name_history_1):

    fig, ax = plt.subplots(1, 2, figsize=(15,5))

    ax[0].set_title('loss')

    ax[0].plot(history.epoch, history.history["loss"], label="Train loss " + name_history)
    ax[0].plot(history.epoch, history.history["val_loss"], label="Validation loss " + name_history)

    ax[0].plot(history_1.epoch, history_1.history["loss"], label="Train loss " + name_history_1)
    ax[0].plot(history_1.epoch, history_1.history["val_loss"], label="Validation loss " + name_history_1)

    ax[1].set_title('Accuracy')

    ax[1].plot(history.epoch, history.history["accuracy"], label="Train Accuracy " + name_history)
    ax[1].plot(history.epoch, history.history["val_accuracy"], label="Validation Accuracy " + name_history)

    ax[1].plot(history_1.epoch, history_1.history["accuracy"], label="Train Accuracy " + name_history_1)
    ax[1].plot(history_1.epoch, history_1.history["val_accuracy"], label="Validation Accuracy " + name_history_1)

    ax[0].legend()
    ax[1].legend()

## 💾 Load Dataset

* This Dataset contains about 5000 pictures of cats and dogs (4000 for training and 1000 for testing).
* Each category will be in a separated folder, `cat-and-dog/training_set/` and `cat-and-dog/test_set` and then in sub folders `cats` and `dogs`.

In [ ]:
# !curl "https://wagon-public-datasets.s3.amazonaws.com/deep_learning_datasets/cat-and-dog.zip" --output cat-and-dog.zip
# !unzip -qn cat-and-dog.zip

In [ ]:
!ls

In [ ]:
!tree -L 3

In [ ]:
train_data_dir = "cat-and-dog-small/training_set/"
test_data_dir = "cat-and-dog-small/test_set/"

## 👀 Explore Dataset

In [ ]:
path_train_cat_4 = f"{train_data_dir}cats/cat.4.jpg"

In [ ]:
img_train_cat_4 = load_img(path_train_cat_4)
img_train_cat_4

In [ ]:
type(img_train_cat_4)

In [ ]:
x_train_cat_4 = img_to_array(img_train_cat_4)/255.
x_train_cat_4

In [ ]:
type(x_train_cat_4)

In [ ]:
x_train_cat_4.shape

In [ ]:
# x_train_cat_3

## 🗂️ "Features" (Images) & "Target" (Folder name)

In [ ]:
BATCH_SIZE = 64

train_ds = image_dataset_from_directory(
    train_data_dir,
    labels = "inferred",
    label_mode = "binary",
    image_size = (224,224),
    batch_size = BATCH_SIZE
)

In [ ]:
val_ds = image_dataset_from_directory(
    test_data_dir,
    labels = "inferred",
    label_mode = "binary",
    image_size = (224,224),
    batch_size = BATCH_SIZE
)

In [ ]:
type(train_ds)

In [ ]:
train_ds.class_names

['cats', 'dogs']

## 🧠 CNN

### (1) Model V1

In [ ]:
def first_model():

    # 1 - ARCHITECTURE

    model = models.Sequential([
        ### INPUT LAYER
        Input(shape=(224,224,3)),

        ### SCALING
        layers.Rescaling(1/255.),
        ### CONVOLUTIONS
        ###### CONV no. 1
        layers.Conv2D(filters=32, kernel_size = (3,3), activation="relu"),
        layers.MaxPool2D(pool_size=(2,2)),
        ###### CONV no. 2
        layers.Conv2D(filters=32, kernel_size = (3,3), activation="relu"),
        layers.MaxPool2D(pool_size=(2,2)),
        ###### CONV no. 3
        layers.Conv2D(filters=64, kernel_size = (3,3), activation="relu"),
        layers.MaxPool2D(pool_size=(2,2)),
        ###### CONV no. 4
        layers.Conv2D(filters=128, kernel_size = (3,3), activation="relu"),
        layers.MaxPool2D(pool_size=(2,2)),
        ### FLATTEN
        layers.Flatten(),
        ### SOME HIDDEN DENSE LAYERS
        layers.Dense(64, activation="relu"),
        ### PREDICTIVE/OUTPUT LAYER
        layers.Dense(1, activation="sigmoid")


    ])

    # 2 - COMPILE
    my_adam = optimizers.Adam(learning_rate = 0.002)

    model.compile(loss="binary_crossentropy", #"categorical_crossentropy"
                  optimizer = my_adam,        #"sparse_categorical_crossentropy"
                  metrics = ["accuracy"])


    return model

In [ ]:
first_model = first_model()

MODEL = "model_1.keras"

modelCheckpoint = callbacks.ModelCheckpoint(MODEL,
                                            monitor="val_loss",
                                            verbose=0,
                                            save_best_only=True)

LRreducer = callbacks.ReduceLROnPlateau(monitor="val_loss",
                                        factor=0.1,
                                        patience=3,
                                        verbose=1,
                                        min_lr=0)

EarlyStopper = callbacks.EarlyStopping(monitor='val_loss',
                                       patience=10,
                                       verbose=0,
                                       restore_best_weights=True)

history_1 = first_model.fit(
        train_ds,
        epochs=30,
        validation_data=val_ds,
        callbacks=[modelCheckpoint, LRreducer, EarlyStopper]
        )

Epoch 1/30


2025-08-08 11:13:05.351433: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


63/63 ━━━━━━━━━━━━━━━━━━━━ 12s 113ms/step - accuracy: 0.5064 - loss: 0.8884 - val_accuracy: 0.5514 - val_loss: 0.6862 - learning_rate: 0.0020
Epoch 2/30
63/63 ━━━━━━━━━━━━━━━━━━━━ 6s 102ms/step - accuracy: 0.5023 - loss: 0.7738 - val_accuracy: 0.5247 - val_loss: 0.7287 - learning_rate: 0.0020
Epoch 3/30
63/63 ━━━━━━━━━━━━━━━━━━━━ 7s 111ms/step - accuracy: 0.5456 - loss: 0.7231 - val_accuracy: 0.4990 - val_loss: 0.8126 - learning_rate: 0.0020
Epoch 4/30
63/63 ━━━━━━━━━━━━━━━━━━━━ 7s 119ms/step - accuracy: 0.5594 - loss: 0.7409 - val_accuracy: 0.6008 - val_loss: 0.6522 - learning_rate: 0.0020
Epoch 5/30
63/63 ━━━━━━━━━━━━━━━━━━━━ 8s 125ms/step - accuracy: 0.5976 - loss: 0.6554 - val_accuracy: 0.6047 - val_loss: 0.6575 - learning_rate: 0.0020
Epoch 6/30
63/63 ━━━━━━━━━━━━━━━━━━━━ 8s 130ms/step - accuracy: 0.6062 - loss: 0.6514 - val_accuracy: 0.6324 - val_loss: 0.6479 - learning_rate: 0.0020
Epoch 7/30
63/63 ━━━━━━━━━━━━━━━━━━━━ 8s 132ms/step - accuracy: 0.5998 - loss: 0.6857 - val_accura

In [ ]:
plot_history(history_1)

### (2) Model V2 with Data Augmentation

In [ ]:
def second_model():

    # 1 - ARCHITECTURE

    model = models.Sequential([
        ### INPUT LAYER
        Input(shape=(224,224,3)),

        ### SCALING
        layers.Rescaling(1/255.),

        ### NEW !!!! DATA AUGMENTATION
        layers.RandomFlip("horizontal"),
        # layers.RandomFlip("vertical"),
        # layers.RandomZoom(2),
        layers.RandomZoom(0.1),
        #layers.RandomTranslation((0.2, 0.2)),
        layers.RandomRotation(0.1),

        ### CONVOLUTIONS
        ###### CONV no. 1
        layers.Conv2D(filters=32, kernel_size = (3,3), activation="relu"),
        layers.MaxPool2D(pool_size=(2,2)),
        ###### CONV no. 2
        layers.Conv2D(filters=32, kernel_size = (3,3), activation="relu"),
        layers.MaxPool2D(pool_size=(2,2)),
        ###### CONV no. 3
        layers.Conv2D(filters=64, kernel_size = (3,3), activation="relu"),
        layers.MaxPool2D(pool_size=(2,2)),
        ###### CONV no. 4
        layers.Conv2D(filters=128, kernel_size = (3,3), activation="relu"),
        layers.MaxPool2D(pool_size=(2,2)),

        ### FLATTEN
        layers.Flatten(),
        ### SOME HIDDEN DENSE LAYERS
        layers.Dense(64, activation="relu"),
        ### PREDICTIVE/OUTPUT LAYER
        layers.Dense(1, activation="sigmoid")


    ])

    # 2 - COMPILE
    my_adam = optimizers.Adam(learning_rate = 0.002)

    model.compile(loss="binary_crossentropy", #"categorical_crossentropy"
                  optimizer = my_adam,        #"sparse_categorical_crossentropy"
                  metrics = ["accuracy"])


    return model

In [ ]:
second_model = second_model()

MODEL = "model_2.keras"

modelCheckpoint = callbacks.ModelCheckpoint(MODEL,
                                            monitor="val_loss",
                                            verbose=0,
                                            save_best_only=True)

LRreducer = callbacks.ReduceLROnPlateau(monitor="val_loss",
                                        factor=0.1,
                                        patience=3,
                                        verbose=1,
                                        min_lr=0)

EarlyStopper = callbacks.EarlyStopping(monitor='val_loss',
                                       patience=10,
                                       verbose=0,
                                       restore_best_weights=True)


In [ ]:
history_2 = second_model.fit(
        train_ds,
        epochs=30,
        validation_data=val_ds,
        callbacks = [modelCheckpoint, LRreducer, EarlyStopper]
        )

Epoch 1/30
63/63 ━━━━━━━━━━━━━━━━━━━━ 12s 160ms/step - accuracy: 0.5160 - loss: 0.8465 - val_accuracy: 0.5000 - val_loss: 0.6943 - learning_rate: 0.0020
Epoch 2/30
63/63 ━━━━━━━━━━━━━━━━━━━━ 11s 168ms/step - accuracy: 0.5029 - loss: 0.6963 - val_accuracy: 0.5336 - val_loss: 0.6916 - learning_rate: 0.0020
Epoch 3/30
63/63 ━━━━━━━━━━━━━━━━━━━━ 11s 167ms/step - accuracy: 0.5399 - loss: 0.6897 - val_accuracy: 0.5000 - val_loss: 0.7244 - learning_rate: 0.0020
Epoch 4/30
63/63 ━━━━━━━━━━━━━━━━━━━━ 10s 160ms/step - accuracy: 0.5100 - loss: 0.7874 - val_accuracy: 0.5267 - val_loss: 2.4999 - learning_rate: 0.0020
Epoch 5/30
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 153ms/step - accuracy: 0.5051 - loss: 2.0618
Epoch 5: ReduceLROnPlateau reducing learning rate to 0.00020000000949949026.
63/63 ━━━━━━━━━━━━━━━━━━━━ 10s 162ms/step - accuracy: 0.5051 - loss: 2.0569 - val_accuracy: 0.5267 - val_loss: 1.8014 - learning_rate: 0.0020
Epoch 6/30
63/63 ━━━━━━━━━━━━━━━━━━━━ 10s 162ms/step - accuracy: 0.5172 - loss: 1.3

In [ ]:
plot_history(history_2)

In [ ]:
plot_compare_history(history_1, "Basic CNN", history_2, "Data Augmentation")

### (3) Model V3 with Transfer Learning

In [ ]:
# vgg16 = VGG16(include_top = False, weights="imagenet", input_shape=(224,224,3))
# vgg16.trainable = False
# vgg16.summary()

In [ ]:
def my_customized_vgg16():
    # PRETRAINED LAYERS
    vgg16 = VGG16(include_top = False, weights="imagenet", input_shape=(224,224,3))
    vgg16.trainable = False
    # MY LAYERS
    flatten = layers.Flatten()
    dense_1 = layers.Dense(32, activation="relu")
    dropout_1 = layers.Dropout(0.3)
    dense_2 = layers.Dense(16, activation="relu")
    dropout_2 = layers.Dropout(0.3)
    predictive = layers.Dense(1, "sigmoid")

    model = models.Sequential([
        vgg16,
        flatten,
        dense_1,
        dropout_1,
        dense_2,
        dropout_2,
        predictive
    ])

    # 2 - COMPILE
    my_adam = optimizers.Adam(learning_rate = 0.002)

    model.compile(loss="binary_crossentropy", #"categorical_crossentropy"
                  optimizer = my_adam,        #"sparse_categorical_crossentropy"
                  metrics = ["accuracy"])


    return model

In [ ]:
third_model = my_customized_vgg16()
third_model.summary()

In [ ]:
third_model = my_customized_vgg16()

MODEL = "model_3.keras"

modelCheckpoint = callbacks.ModelCheckpoint(MODEL,
                                            monitor="val_loss",
                                            verbose=0,
                                            save_best_only=True)

LRreducer = callbacks.ReduceLROnPlateau(monitor="val_loss",
                                        factor=0.1,
                                        patience=3,
                                        verbose=1,
                                        min_lr=0)

EarlyStopper = callbacks.EarlyStopping(monitor='val_loss',
                                       patience=10,
                                       verbose=0,
                                       restore_best_weights=True)

history_3 = third_model.fit(
        train_ds,
        epochs=30,
        validation_data=val_ds,
        callbacks = [modelCheckpoint, LRreducer, EarlyStopper])

In [ ]:
plot_history(history_3)

In [ ]:
plot_compare_history(history_1, "Basic CNN", history_3, "Transfer Learning")

## 🔮 Predictions

In [ ]:
from PIL import Image
import requests
from io import BytesIO

def getImage(url):
    '''Grabs an image based on its URL, and resize it
    '''
    response = requests.get(url)
    img = Image.open(BytesIO(response.content))
    plt.imshow(img)
    img = img.resize((224, 224))
    return img

def predictImage(url, model):
    '''Takes an image and a model
    '''
    img = getImage(url)
    img = img_to_array(img)
    img = img.reshape((-1, 224, 224, 3))
    res = model.predict(img)[0][0]
    if(res < 0.5):
        animal = "cat"
        prob = 1-res
    if(res >= 0.5):
        animal = "dog"
        prob = res

    print("Animal : ", animal)
    print("probability = ",prob)

In [ ]:
cat = "https://www.alleycat.org/wp-content/uploads/2019/03/FELV-cat.jpg"
cat

In [ ]:
predictImage(cat, third_model)

In [ ]:
dog = "https://images.sudouest.fr/2018/04/14/5ace461a66a4bd2b1780a0dd/widescreen/1000x500/on-ignore-si-le-chihuahua-a-deserte-ou-non-les-locaux-de-la-clinique.jpg?v1"
dog

In [ ]:
predictImage(dog, third_model)

In [ ]:
davys_cat = ""

In [ ]:
predictImage(trump, third_model)